In [1]:
# Load Libraries
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata
import pandas as pd
import numpy as np


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-monitoring 2.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requi

/tmp/ipykernel_3812/2346600564.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [14]:
api_key = 'API'

In [22]:
import google.generativeai as genai
from google.colab import userdata

# Configure Gemini API
genai.configure(api_key=api_key)

# Define our two models: The fast/cheap traffic cop, and the heavy/expensive genius
fast_model = genai.GenerativeModel('gemini-3.1-flash-lite')
heavy_model = genai.GenerativeModel('gemini-3.5-flash-lite')


In [16]:
import pandas as pd
prompts = pd.read_csv("/content/sample_prompts_classification-v2.csv")
print(prompts['Prompt'])

0                  What is the capital of Saudi Arabia?
1      Summarize this short paragraph in two sentences.
2                               Convert 100 USD to EUR.
3                 What is the chemical symbol for gold?
4         Translate 'Hello, how are you?' into Spanish.
5                  Give me 5 synonym words for 'happy'.
6                         Who wrote 'Romeo and Juliet'?
7                                   What is 15% of 200?
8        List the planets in our solar system in order.
9                     How many days are in a leap year?
10            Define the term 'photosynthesis' briefly.
11            Fix the spelling errors in this sentence.
12      What is the freezing point of water in Celsius?
13           Give me a quick recipe for scrambled eggs.
14        Who was the first person to walk on the Moon?
15           What is the tallest mountain in the world?
16                       How many continents are there?
17                      What is the square root 

In [17]:
user_context_tree = {
    "career": {
        "education": "MSc in Business Studies (MIS Concentration), BSc in MIS (First Honors).",
        "certifications": "Data Analyst in Power BI completed. Microsoft PL-300 and SAP YPP (Ariba/Fieldglass) in progress.",
        "goals": "Pursuing enterprise Graduate Development Programs (GDP) as a Data Analyst or AI Architect."
    },
    "fitness": {
        "routine": "3-5 days weekly. Outdoor walking, treadmill cardio, and HIIT resistance training.",
        "nutrition": "High-protein meals, cottage cheese, blended fruit/Greek yogurt oatmeal."
    },
    "gaming": {
        "setup": "Acer Predator Helios 300, Samsung Galaxy S24 Ultra.",
        "games": "Mobile Legends, Valorant, EA Sports FC, Genshin Impact."
    }
}
print(user_context_tree)

{'career': {'education': 'MSc in Business Studies (MIS Concentration), BSc in MIS (First Honors).', 'certifications': 'Data Analyst in Power BI completed. Microsoft PL-300 and SAP YPP (Ariba/Fieldglass) in progress.', 'goals': 'Pursuing enterprise Graduate Development Programs (GDP) as a Data Analyst or AI Architect.'}, 'fitness': {'routine': '3-5 days weekly. Outdoor walking, treadmill cardio, and HIIT resistance training.', 'nutrition': 'High-protein meals, cottage cheese, blended fruit/Greek yogurt oatmeal.'}, 'gaming': {'setup': 'Acer Predator Helios 300, Samsung Galaxy S24 Ultra.', 'games': 'Mobile Legends, Valorant, EA Sports FC, Genshin Impact.'}}


In [18]:
def get_context_shard(prompt):
    """Uses the fast model to decide which branch of the context tree is relevant."""
    instruction = f"""
    Analyze the user prompt and categorize it into exactly one of these domains: 'career', 'fitness', 'gaming', or 'general'.
    Output ONLY the single word of the category.
    User Prompt: {prompt}
    """
    response = fast_model.generate_content(instruction)
    category = response.text.strip().lower()

    # Retrieve the specific nested dictionary or fallback to empty string
    shard = user_context_tree.get(category, "No specific context needed.")
    return category, shard

In [19]:
def ai_complexity_router(prompt):
    """Uses the fast model to evaluate the compute-cost/complexity of the prompt."""
    instruction = f"""
    Analyze this user prompt for required compute resources.
    If it requires complex reasoning, advanced coding, data analysis, or deep generation, output 'COMPLEX'.
    If it is a simple factual question, greeting, or basic retrieval, output 'SIMPLE'.
    Output ONLY the single word 'SIMPLE' or 'COMPLEX'.
    User Prompt: {prompt}
    """
    response = fast_model.generate_content(instruction)
    return response.text.strip().upper()

In [20]:
def process_smart_query(prompt):
    """The master orchestrator combining context sharding and smart routing."""
    print(f"--- Processing New Prompt ---")
    print(f"User: {prompt}\n")

    # 1. Fetch only the relevant context (saving tokens)
    category, shard = get_context_shard(prompt)
    print(f"[System] Selected Context Shard: {category.upper()}")

    # 2. Judge the complexity
    route = ai_complexity_router(prompt)
    print(f"[System] AI Router Classification: {route}")

    # 3. Build the highly-efficient prompt payload
    final_payload = f"User Context: {shard}\n\nUser Request: {prompt}"

    # 4. Route to the appropriate model based on the AI's classification
    if route == "COMPLEX":
        print("[System] Routing to GEMINI-3.5-FLASH-Lite (Heavy Compute)")
        final_response = heavy_model.generate_content(final_payload)
    else:
        print("[System] Routing to GEMINI-3.1-FLASH-Lite (Light Compute)")
        final_response = fast_model.generate_content(final_payload)

    print(f"\nResponse:\n{final_response.text}\n")
    print("-" * 40)

In [23]:
from numpy import random
# Randomly select a prompt using NumPy
selected_prompt = random.choice(prompts['Prompt'])
print(selected_prompt)

# Run the program
process_smart_query(selected_prompt)

Name the three primary colors.
--- Processing New Prompt ---
User: Name the three primary colors.

[System] Selected Context Shard: GENERAL
[System] AI Router Classification: SIMPLE
[System] Routing to GEMINI-3.1-FLASH-Lite (Light Compute)

Response:
The three primary colors (in the traditional RYB color model used in art) are **red, yellow, and blue**.

----------------------------------------
